##### 1. Objective

##### 10 - Fine-Tune Vision Model

##### Objective

Train a pretrained ResNet18 model for product-defect image
classification using a two-stage transfer-learning strategy.

##### Stage 1 — Feature Extraction

The pretrained ResNet18 backbone remains frozen.

Only the new classification layer:     Linear(512 → 2) is trained.

##### Stage 2 — Fine-Tuning

The final ResNet18 feature block (`layer4`) is unfrozen.

The model can then make small adjustments to higher-level pretrained
visual features while continuing to train the classifier.

##### Model Selection

Validation loss is used to compare the best Stage 1 model with the
best Stage 2 model.

The model with the lower validation loss is retained.

The untouched test set is not used for training or model selection.

##### 2. Before coding: understand the two stages

##### Stage 1 — Feature extraction

``` text

Casting image
      ↓
ResNet18 backbone
      🔒
pretrained weights unchanged
      ↓
512 feature values
      ↓
Linear(512 → 2)
      🔓
new weights learn
      ↓
def_front / ok_front

```
Only:

fc.weight
fc.bias

learn.

##### Stage 2 — Fine-tuning

After the classifier has learned something useful, we'll unfreeze the last ResNet block:

``` text

Earlier ResNet layers
        🔒
        ↓
Generic visual features
        ↓
layer4
        🔓
        ↓
Casting-related adaptation
        ↓
Linear(512 → 2)
        🔓

```
We won't unfreeze the entire ResNet18 immediately.

Why?

Earlier layers tend to capture more generic patterns such as:

- edges
- lines
- basic textures

Later layers contain more task-specific higher-level representations.

So adjusting later layers first is a reasonable transfer-learning strategy.

##### 3. Imports

In [0]:
from copy import deepcopy

import mlflow
import mlflow.pytorch

import torch

from torch import nn
from torch.optim import Adam

from src.data_preparation import (
    create_image_dataloaders,
)

from src.model_training import (
    train_model,
)

from src.project_config import (
    FEATURE_EXTRACTION_EPOCHS,
    FEATURE_EXTRACTION_LEARNING_RATE,
    FINE_TUNING_EPOCHS,
    FINE_TUNING_LEARNING_RATE,
    MLFLOW_EXPERIMENT_NAME,
)

from src.vision_model import (
    build_resnet18_classifier,
)

##### 4. Select Device

In [0]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device,
)

##### 5. Create Pretrained-Compatible DataLoaders

In [0]:
data = create_image_dataloaders(
    preprocessing_mode="pretrained",
)

print(
    "Classes:",
    data.class_names,
)

print(
    "Class mapping:",
    data.class_to_idx,
)

print(
    "Train samples:",
    len(data.train_dataset),
)

print(
    "Validation samples:",
    len(data.val_dataset),
)

print(
    "Test samples:",
    len(data.test_dataset),
)

##### 6. Build Frozen ResNet18

In [0]:
model = build_resnet18_classifier(
    num_classes=data.num_classes,
    freeze_backbone=True,
)

model = model.to(
    device
)

In [0]:
model

In [0]:
for name, parameter in model.named_parameters():
    print(
        name,
        parameter.shape,
        parameter.requires_grad)


In [0]:
stage_1_trainable_names = [
    name
    for name, parameter
    in model.named_parameters()
    if parameter.requires_grad
]

stage_1_trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    "Stage 1 trainable parameter tensors:"
)

for name in stage_1_trainable_names:
    print(
        name
    )

print(
    "\nStage 1 trainable individual parameters:",
    f"{stage_1_trainable_parameters:,}",
)

In [0]:
assert stage_1_trainable_names == [
    "fc.weight",
    "fc.bias",
]

assert (
    stage_1_trainable_parameters
    == 1_026
)

##### 7. Define Loss Function

In [0]:
criterion = (
    nn.CrossEntropyLoss()
)

##### 8. Configure MLflow experiment

In [0]:
mlflow.set_experiment(
    MLFLOW_EXPERIMENT_NAME
)

We'll use one MLflow run containing both stages:

ResNet18 Transfer Learning Run
        │
        ├── Stage 1 metrics
        │
        ├── Stage 2 metrics
        │
        ├── parameters
        │
        └── selected final model

That makes the complete training process one experiment run.

##### 9. Start MLflow run

In [0]:
# Close any run left active in the current notebook session.

if mlflow.active_run() is not None:
    mlflow.end_run()


run = mlflow.start_run(
    run_name="resnet18_transfer_learning"
)

run_id = (
    run.info.run_id
)

print(
    "MLflow Run ID:",
    run_id,
)

The remaining Stage 1 and Stage 2 code should be inside this with block.

#####  10. Log experiment configuration

In [0]:
mlflow.log_params(
    {
        "architecture": "resnet18",
        "pretrained": True,
        "num_classes": data.num_classes,
        "feature_extraction_epochs": FEATURE_EXTRACTION_EPOCHS,
        "feature_extraction_learning_rate": FEATURE_EXTRACTION_LEARNING_RATE,
        "fine_tuning_epochs": FINE_TUNING_EPOCHS,
        "fine_tuning_learning_rate": FINE_TUNING_LEARNING_RATE,
        "fine_tuned_block": "layer4",
        "model_selection_metric":
            "validation_loss",
    }
)

##### 11. Stage 1 — Feature Extraction Training

The pretrained ResNet18 backbone remains frozen.

Only the newly initialized classification layer is trained:

    Linear(512 → 2)

The backbone continues to calculate pretrained visual features for
each image, but its learned weights are not updated.

In [0]:
feature_extraction_optimizer = Adam(
    (
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ),
    lr=FEATURE_EXTRACTION_LEARNING_RATE,
)

The optimizer receives only:

- fc.weight
- fc.bias

Now train:

In [0]:
feature_result = train_model(
    model=model,
    train_loader=data.train_loader,
    val_loader=data.val_loader,
    criterion=criterion,
    optimizer=feature_extraction_optimizer,
    device=device,
    num_epochs=FEATURE_EXTRACTION_EPOCHS,
)

reusable train_model() now handles:

Epoch 1
   ↓
train_one_epoch()
   ↓
validate_one_epoch()
   ↓
save metrics
   ↓
check best model

Epoch 2
...

Epoch 3
...

Restore best Stage-1 model

##### 12. Inspect Stage 1 result

In [0]:
print(
    "\nStage 1 Best Epoch:",
    feature_result.best_epoch,
)

print(
    "Stage 1 Best Validation Loss:",
    f"{feature_result.best_val_loss:.4f}",
)

print(
    "Stage 1 Best Validation Accuracy:",
    f"{feature_result.best_val_accuracy:.4f}",
)

##### 13. Log Stage 1 epoch metrics

In [0]:
#This loop does not train the model again.It simply persists the history that train_model() already collected.

for epoch in range(
    FEATURE_EXTRACTION_EPOCHS
):

    step = (
        epoch + 1
    )

    mlflow.log_metric(
        "feature_train_loss",
        feature_result.history[
            "train_loss"
        ][epoch],
        step=step,
    )

    mlflow.log_metric(
        "feature_train_accuracy",
        feature_result.history[
            "train_accuracy"
        ][epoch],
        step=step,
    )

    mlflow.log_metric(
        "feature_val_loss",
        feature_result.history[
            "val_loss"
        ][epoch],
        step=step,
    )

    mlflow.log_metric(
        "feature_val_accuracy",
        feature_result.history[
            "val_accuracy"
        ][epoch],
        step=step,
    )

In [0]:
#Also log the best Stage 1 values:

mlflow.log_metric(
    "feature_best_val_loss",
    feature_result.best_val_loss,
)

mlflow.log_metric(
    "feature_best_val_accuracy",
    feature_result.best_val_accuracy,
)

mlflow.log_metric(
    "feature_best_epoch",
    feature_result.best_epoch,
)

##### 14. Preserve the best Stage 1 model

In [0]:
stage_1_best_state = deepcopy(
    model.state_dict()
)


This step is critical.

Remember that train_model() has already restored:

Best Stage-1 model

Therefore:

    stage_1_best_state = deepcopy(
        model.state_dict()
    )

Now we have an independent in-memory copy of the best Stage 1 weights.

We need it because Stage 2 is about to modify the model.

##### 15. Stage 2 — Fine-Tuning Training

The best Stage 1 model becomes the starting point for Stage 2.

The final ResNet18 feature block (`layer4`) is now unfrozen.

Earlier pretrained layers remain frozen.

This allows higher-level visual features to make small adaptations to
the casting-defect dataset.

In [0]:
#Unfreeze layer4:

for parameter in (
    model.layer4.parameters()
):
    parameter.requires_grad = True

Input image
   ↓
conv1
   ↓
maxpool
   ↓
layer1
   ↓
layer2
   ↓
layer3
   ↓
layer4          ← last convolutional / feature-extraction block
   ↓
avgpool
   ↓
Flatten
   ↓
fc              ← final classification layer
Linear(512 → 2)
   ↓
2 logits

Earlier                                      Later

conv1 → layer1 → layer2 → layer3 → layer4 → fc
  │        │         │         │        │
basic     lower     more      higher    most
visual    visual    complex   level     task-relevant
patterns  patterns   patterns  features features

Conceptually:

conv1      🔒
layer1     🔒
layer2     🔒
layer3     🔒

layer4     🔓

fc         🔓

##### 16. Verify Stage 2 trainable parameters

In [0]:
stage_2_trainable_names = [
    name
    for name, parameter
    in model.named_parameters()
    if parameter.requires_grad
]

stage_2_trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    "\nStage 2 trainable parameter tensors:"
)

for name in stage_2_trainable_names:
    print(
        name
    )

print(
    "\nStage 2 trainable individual parameters:",
    f"{stage_2_trainable_parameters:,}",
)

In [0]:
assert any(
    name.startswith(
        "layer4."
    )
    for name
    in stage_2_trainable_names
)

assert "fc.weight" in (
    stage_2_trainable_names
)

assert "fc.bias" in (
    stage_2_trainable_names
)

assert (
    stage_2_trainable_parameters
    > stage_1_trainable_parameters
)

##### 17. Create Stage 2 optimizer

In [0]:
#Because we've changed which parameters are trainable, create a new optimizer:

fine_tuning_optimizer = Adam(
    (
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ),
    lr=FINE_TUNING_LEARNING_RATE,
)

##### 18. Fine-tune

In [0]:
fine_tuning_result = train_model(
    model=model,
    train_loader=data.train_loader,
    val_loader=data.val_loader,
    criterion=criterion,
    optimizer=fine_tuning_optimizer,
    device=device,
    num_epochs=FINE_TUNING_EPOCHS,
)

#Again, train_model() restores the best Stage 2 model automatically.

##### 19. Inspect Stage 2 result

In [0]:
print(
    "\nStage 2 Best Epoch:",
    fine_tuning_result.best_epoch,
)

print(
    "Stage 2 Best Validation Loss:",
    f"{fine_tuning_result.best_val_loss:.4f}",
)

print(
    "Stage 2 Best Validation Accuracy:",
    f"{fine_tuning_result.best_val_accuracy:.4f}",
)

##### 20. Log Stage 2 epoch metrics

In [0]:
#Again, this does not train again.

for epoch in range(
    FINE_TUNING_EPOCHS
):

    step = (
        epoch + 1
    )

    mlflow.log_metric(
        "fine_tune_train_loss",
        fine_tuning_result.history[
            "train_loss"
        ][epoch],
        step=step,
    )

    mlflow.log_metric(
        "fine_tune_train_accuracy",
        fine_tuning_result.history[
            "train_accuracy"
        ][epoch],
        step=step,
    )

    mlflow.log_metric(
        "fine_tune_val_loss",
        fine_tuning_result.history[
            "val_loss"
        ][epoch],
        step=step,
    )

    mlflow.log_metric(
        "fine_tune_val_accuracy",
        fine_tuning_result.history[
            "val_accuracy"
        ][epoch],
        step=step,
    )

In [0]:
mlflow.log_metric(
    "fine_tune_best_val_loss",
    fine_tuning_result.best_val_loss,
)

mlflow.log_metric(
    "fine_tune_best_val_accuracy",
    fine_tuning_result.best_val_accuracy,
)

mlflow.log_metric(
    "fine_tune_best_epoch",
    fine_tuning_result.best_epoch,
)

##### 21. Preserve the best Stage 2 model

In [0]:
stage_2_best_state = deepcopy(
    model.state_dict()
)

stage_1_best_state
        ↓
Best feature-extraction model


stage_2_best_state
        ↓
Best fine-tuned model

##### 22. Compare Stage 1 vs Stage 2

In [0]:
if (
    fine_tuning_result.best_val_loss
    < feature_result.best_val_loss
):

    selected_stage = (
        "fine_tuning"
    )

    selected_best_epoch = (
        fine_tuning_result.best_epoch
    )

    selected_val_loss = (
        fine_tuning_result.best_val_loss
    )

    selected_val_accuracy = (
        fine_tuning_result.best_val_accuracy
    )

    model.load_state_dict(
        stage_2_best_state
    )

else:

    selected_stage = (
        "feature_extraction"
    )

    selected_best_epoch = (
        feature_result.best_epoch
    )

    selected_val_loss = (
        feature_result.best_val_loss
    )

    selected_val_accuracy = (
        feature_result.best_val_accuracy
    )

    model.load_state_dict(
        stage_1_best_state
    )

In [0]:
print(
    "\nSelected Stage:",
    selected_stage,
)

print(
    "Selected Best Epoch:",
    selected_best_epoch,
)

print(
    "Selected Validation Loss:",
    f"{selected_val_loss:.4f}",
)

print(
    "Selected Validation Accuracy:",
    f"{selected_val_accuracy:.4f}",
)

##### 23. Log selected model information

In [0]:
mlflow.log_param(
    "selected_stage",
    selected_stage,
)

mlflow.log_metric(
    "selected_best_epoch",
    selected_best_epoch,
)

mlflow.log_metric(
    "selected_val_loss",
    selected_val_loss,
)

mlflow.log_metric(
    "selected_val_accuracy",
    selected_val_accuracy,
)

##### 24. Log the selected model to MLflow

In [0]:
mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path="model",
)

##### 25. Print persistent model reference

In [0]:
model_uri = (
    f"runs:/{run_id}/model"
)

print(
    "\nMLflow Run ID:",
    run_id,
)

print(
    "Model URI:",
    model_uri,
)

In [0]:
mlflow.end_run()

assert mlflow.active_run() is None

print(
    "MLflow run completed successfully."
)

##### 26. Final sanity check

In [0]:
model.eval()

sample_images, sample_labels = next(
iter(
    data.val_loader
)
)

sample_images = sample_images[
:4
].to(
device
)

with torch.no_grad():

    sample_logits = model(
        sample_images
    )

    print(
    "Sample input shape:",
    sample_images.shape,
    )

    print(
    "Sample logits shape:",
    sample_logits.shape,
    )

    print(
    "Sample logits:"
    )

    print(
    sample_logits
)

##### 27. Display complete training history

In [0]:
print(
    "Feature Extraction History:"
)

print(
    feature_result.history
)

print(
    "\nFine-Tuning History:"
)

print(
    fine_tuning_result.history
)

##### 🔍 Explainable AI (XAI)

Feature extraction keeps the pretrained visual representation fixed,
while fine-tuning allows the final ResNet18 feature block to adapt to
the casting-defect dataset.

This adaptation can change which visual patterns contribute to the
classification.

A later notebook will use Grad-CAM to inspect which image regions
influence predictions made by the selected model.

Grad-CAM indicates model attention/influence regions; it does not prove
the physical location of a defect because this dataset contains
image-level class labels rather than defect-location annotations.

##### ⚖️ Responsible AI (RAI)

Fine-tuning increases the model's ability to adapt to the target
dataset, but greater adaptation does not automatically imply greater
reliability.

The validation set is used to compare feature extraction and
fine-tuning.

The untouched test set is not used to choose:

- learning rates
- epoch counts
- fine-tuning strategy
- selected model stage

This helps preserve the test set as an independent estimate of final
model performance.

Later Responsible AI analysis will examine class-level performance,
missed defects, false defect alarms, confidence, failure cases, and
appropriate human oversight.

##### Key Learnings

1. Transfer learning reuses visual representations learned by a
   pretrained model.

2. During feature extraction, the pretrained ResNet18 backbone is
   frozen and only the new classifier learns.

3. The 512 feature values vary by image, but the frozen backbone
   weights that generate those features remain unchanged during
   Stage 1.

4. The new `Linear(512 → 2)` classifier begins with newly initialized
   weights.

5. During fine-tuning, `layer4` and the classifier become trainable.

6. Fine-tuning uses a smaller learning rate because pretrained weights
   already contain useful information.

7. Feature extraction and fine-tuning can use different numbers of
   epochs because they train different sets of parameters.

8. Training metrics are retained for every epoch.

9. A separate permanent model file is not required for every epoch.
   The best model is selected using validation loss.

10. Fine-tuning is not automatically assumed to be better than feature
    extraction. The best validation model is retained.

11. The test set remains untouched during training and model
    selection.

12. MLflow persists the epoch metrics, experiment configuration, and
    selected model so later notebooks do not depend on notebook memory.

##### Conclusion

A pretrained ResNet18 model was adapted to the product-defect
classification task using two-stage transfer learning.

Stage 1 trained only the newly initialized classifier while keeping
the pretrained backbone frozen.

Stage 2 unfroze the final ResNet18 feature block and performed
fine-tuning using a smaller learning rate.

The best feature-extraction and fine-tuning models were compared using
validation loss, and the better validation model was retained.

The complete training history and selected model were persisted in
MLflow.

The untouched test set remains available for independent model
evaluation and comparison with the custom CNN.

In [0]:
mlflow.active_run()

In [0]:
mlflow.end_run()